<a href="https://colab.research.google.com/github/SeiDra/lending-club-prediction/blob/main/du_sda_ml2_P4_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PROJET 7 : Loan Default Prediction
Partie N°4 — Modélisation & Évaluation

Contenu :
- Import des données transformées (sortie du P3)
- Gestion du déséquilibre (Undersampling)
- Feature Selection
- Évaluation LightGBM (Cross-Validation 5-fold)
- Optimisation du modèle
- Évaluation finale sur le Test Set (une seule fois)
- Interprétabilité et conclusions

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as sps
import seaborn as sns

from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, f1_score, precision_score,
                              recall_score, roc_auc_score, roc_curve,
                              precision_recall_curve)
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

In [ ]:
train_X = pd.read_parquet("DATA/03_train_X.parquet")
train_y = pd.read_parquet("DATA/03_train_y.parquet").squeeze()
test_X = pd.read_parquet("DATA/03_test_X.parquet")
test_y = pd.read_parquet("DATA/03_test_y.parquet").squeeze()

# Lecture du nom de la cible (pour les fonctions utilitaires)
with open("CONFIG/target_config.txt", "r") as f:
    target_col = f.read().strip()

print(f"Train X : {train_X.shape} | Train y : {train_y.shape}")
print(f"Test  X : {test_X.shape}  | Test  y : {test_y.shape}")
print(f"\nRatio défaut train : {train_y.mean():.2%}")
print(f"Ratio défaut test  : {test_y.mean():.2%}")

### Gestion du déséquilibre des classes

**Pourquoi privilégier l'Undersampling plutôt que SMOTE ou les paramètres de poids (`is_unbalance=True`, `scale_pos_weight`) ?**

Bien que des algorithmes comme LightGBM proposent des paramètres intégrés (`is_unbalance=True` ou `scale_pos_weight`) et que la génération de données synthétiques (comme SMOTE) soit populaire, nous avons opté pour un **Undersampling (sous-échantillonnage) de la classe majoritaire** pour plusieurs raisons clés, particulièrement adaptées au contexte de Lending Club :

1. **Volume des données et temps de calcul :** Le jeu de données est massif (plus d'un million de lignes). L'undersampling réduit considérablement la taille du jeu de données d'entraînement. Cela accélère drastiquement l'apprentissage et surtout les étapes d'optimisation des hyperparamètres (GridSearch/RandomSearch). À l'inverse, SMOTE augmenterait encore la taille du dataset, rendant l'entraînement extrêmement long et gourmand en mémoire.
2. **Redondance de la classe majoritaire :** La classe majoritaire (prêts remboursés) est tellement représentée (80% des données) qu'en supprimer une partie de manière aléatoire n'entraîne pas de perte d'information significative. Le modèle disposera toujours de suffisamment d'exemples pour comprendre le profil d'un bon payeur.
3. **Risque de bruit avec SMOTE :** SMOTE génère des échantillons synthétiques par interpolation spatiale. Sur des données tabulaires complexes mêlant variables continues et variables catégorielles encodées, cette interpolation peut créer des individus irréalistes (du bruit) et favoriser le surapprentissage (overfitting). L'undersampling n'utilise que des données 100% réelles.
4. **Clarté du signal :** Utiliser `scale_pos_weight` ou `is_unbalance=True` conserve toute la masse de la classe majoritaire mais pénalise plus lourdement les erreurs sur la classe minoritaire. Avec un déséquilibre fort sur un très gros volume, le nettoyage pur et simple de la classe majoritaire permet souvent au modèle d'isoler un signal plus clair et plus distinct entre les deux classes sans biaiser artificiellement la fonction de coût.

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

undersampler = RandomUnderSampler(random_state=42)
# On crée un jeu d'entraînement rééquilibré
train_X_model, train_y_model = undersampler.fit_resample(train_X, train_y)

print(f"Undersampling : {train_X_model.shape}")
print(train_y_model.value_counts())

In [ ]:
# Méthode : Feature Importance via un modèle LightGBM rapide
from lightgbm import LGBMClassifier

# Entraîner un modèle rapide pour obtenir les importances
lgbm_quick = LGBMClassifier(
    n_estimators=100,
    random_state=42,
    verbose=-1
)
lgbm_quick.fit(train_X_model, train_y_model)

# Extraire et afficher les importances
importances = pd.Series(lgbm_quick.feature_importances_, index=train_X_model.columns)
importances = importances.sort_values(ascending=False)

plt.figure(figsize=(10, 8))
importances.head(20).plot(kind='barh')
plt.title("Top 20 Features par Importance (LightGBM)")
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Sélectionner les features avec importance > 0 (ou un seuil personnalisé)
N_TOP_FEATURES = 15
vars_final = importances.head(N_TOP_FEATURES).index.tolist()
print(f"\n{N_TOP_FEATURES} features retenues : {vars_final}")

# Filtrer train et test
train_X_selected = train_X_model[vars_final]
test_X_selected = test_X[vars_final]

Pour justifier le choix de **LightGBM** par rapport aux autres modèles que nous avons testés (Régression Logistique, Arbre de Décision, KNN, Random Forest, Naive Bayes, XGBoost, Gradient Boosting, Réseau de Neurones), nous avons structurer notre argumentaire autour de ces points clés, particulièrement adaptés au jeu de données **Lending Club** (qui est volumineux et tabulaire) :

**1. Vitesse d'entraînement et efficacité mémoire (Le point fort face à XGBoost et Random Forest)**
*   Contrairement à XGBoost (dans son fonctionnement classique) ou Random Forest qui font croître les arbres niveau par niveau (*level-wise*), LightGBM utilise une croissance par feuille (*leaf-wise*). De plus, il utilise une technique basée sur des histogrammes pour regrouper les valeurs continues.
*   **Résultat :** Il est significativement **plus rapide** à entraîner et consomme **beaucoup moins de mémoire**. Sur un jeu de données massif comme celui de Lending Club, ce gain de temps est crucial pour pouvoir itérer et optimiser les hyperparamètres rapidement.

**2. Supériorité sur les données tabulaires (Face à KNN, SVM, Naïve Bayes et Réseaux de Neurones)**
*   Pour des données structurées/tabulaires contenant un mélange de variables continues et catégorielles (revenus, durée du prêt, taux, etc.), les algorithmes de boosting de gradient (comme LightGBM et XGBoost) dominent historiquement les compétitions de type Kaggle. Ils performent généralement mieux que les réseaux de neurones complexes ou les modèles basés sur la distance comme le KNN (très lent et sensible à la malédiction de la dimensionnalité).


**3. Interprétabilité du modèle (Face aux Réseaux de Neurones)**
*   Dans le domaine du risque de crédit, l'interprétabilité est une exigence métier et parfois légale. Si un modèle de réseau de neurones (`MLPClassifier`) agit comme une "boîte noire", LightGBM permet de calculer très facilement l'**importance des features** (savoir que l'historique de crédit ou le taux d'endettement a été déterminant dans le refus d'un prêt). Il est aussi parfaitement compatible avec des outils de type SHAP values pour expliciter localement des décisions.

**4. Robustesse face au déséquilibre des classes**
*   Le cas d'usage de Lending Club consiste généralement à prédire la probabilité de défaut. Il s'agit de jeux de données où les "défauts de paiement" sont minoritaires face aux prêts remboursés.
*   LightGBM gère nativement le déséquilibre des classes de façon très performante, notamment grâce aux poids (paramètres comme `is_unbalance` ou `scale_pos_weight`).

**En résumé :**
**LightGBM offre le meilleur compromis de notre benchmark** : il atteint des performances prédictives de pointe (supérieures à Random Forest et XGBoost), tout en divisant les temps de calcul de manière drastique, et ce de manière parfaitement transparente et interprétable pour un contexte financier.

In [ ]:
# Optimisation de LightGBM avec RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10, -1],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'num_leaves': [15, 31, 63, 127],
    'min_child_samples': [10, 20, 50],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
}

search = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    param_distributions=param_dist,
    n_iter=30,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='average_precision',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Entraîner sur le jeu d'entraînement rééquilibré (features sélectionnées)
search.fit(train_X_selected, train_y_model)

print(f"\nMeilleurs paramètres : {search.best_params_}")
print(f"Meilleure Average Precision (CV) : {search.best_score_:.4f}")

best_model = search.best_estimator_

In [ ]:
from sklearn.metrics import fbeta_score, average_precision_score

# On établit l'évaluation finale sur le test set
y_pred = best_model.predict(test_X_selected)
y_pred_proba = best_model.predict_proba(test_X_selected)[:, 1]

# Calcul des métriques essentielles
recall_classe1 = recall_score(test_y, y_pred)
f2_score = fbeta_score(test_y, y_pred, beta=2)
auprc = average_precision_score(test_y, y_pred_proba)

# KS Statistic
mask = test_y.astype(bool).values
ks_stat = sps.ks_2samp(y_pred_proba[mask], y_pred_proba[~mask])[0]

print("=" * 60)
print("RÉSUMÉ DES MÉTRIQUES CLÉS - TEST SET")
print("=" * 60)
print(f"🔹 AUPRC (Average Precision) : {auprc:.4f} → Robustesse globale du modèle (indépendant du déséquilibre).")
print(f"🔹 Recall (Classe 1)         : {recall_classe1:.4f} → Protection du capital : taux de détection des vrais défauts.")
print(f"🔹 F2-Score                  : {f2_score:.4f} → Métrique métier : accorde 2x plus de poids au Recall qu'à la Précision.")
print(f"🔹 KS Statistic              : {ks_stat:.4f} → Séparation des distributions (Bon vs Mauvais crédit).")
print("=" * 60)

# Courbe Precision-Recall (pour appui visuel de l'AUPRC)
precision_curve, recall_curve, _ = precision_recall_curve(test_y, y_pred_proba)
plt.figure(figsize=(6, 4))
plt.plot(recall_curve, precision_curve, label=f'AUPRC = {auprc:.4f}', color='darkorange')
plt.xlabel('Recall (Détection des défauts)')
plt.ylabel('Precision (Fiabilité de l\'alerte)')
plt.title('Courbe Precision-Recall — Test Set')
plt.legend()
plt.tight_layout()
plt.show()

# Matrice de confusion rapide (pour la lisibilité)
cm = confusion_matrix(test_y, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Good', 'Bad'], yticklabels=['Good', 'Bad'])
plt.xlabel('Prédiction')
plt.ylabel('Réalité')
plt.title('Matrice de Confusion')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_estimator(best_model, test_X_selected, test_y)
plt.plot([0, 1], [0, 1], color='red', linestyle='--') # Ligne de hasard
plt.title("Courbe ROC - Détection de Défaut de Crédit")
plt.show()

In [ ]:
from sklearn.metrics import precision_score, roc_auc_score, classification_report

# 1. Calcul des prédictions (classes et probabilités)
# Remplace 'X_test' et 'y_test' par les noms de tes variables si elles sont différentes
y_pred = best_model.predict(test_X_selected)
y_proba = best_model.predict_proba(test_X_selected)[:, 1]

# 2. Calcul des métriques spécifiques
precision = precision_score(test_y, y_pred)
auc_roc = roc_auc_score(test_y, y_proba)

# 3. Affichage propre
print(f"========================================")
print(f"   RÉSULTATS FINAUX DU MODÈLE LGBM")
print(f"========================================")
print(f"Précision (Classe 1 - Bad Loan) : {precision:.4f}")
print(f"AUC-ROC Score                   : {auc_roc:.4f}")
print(f"========================================")

# Afficher le rapport complet pour voir la précision de la classe 0 aussi
print("\nDétails par classe :")
print(classification_report(test_y, y_pred))

In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

# Calcul des Précisions, Recalls et Seuils
precisions, recalls, thresholds = precision_recall_curve(test_y, y_proba)

# 1. Tracer le graphique Precision / Recall selon le seuil
plt.figure(figsize=(9, 5))
# On trace jusqu'à l'avant-dernier élément car la taille de thresholds est len(precisions) - 1
plt.plot(thresholds, precisions[:-1], "b--", label="Précision (Fiabilité de l'alerte)", linewidth=2)
plt.plot(thresholds, recalls[:-1], "g-", label="Recall (Détection absolue)", linewidth=2)

# Ligne verticale indiquant le seuil standard par défaut à 0.5
plt.axvline(x=0.5, color='red', linestyle=':', label='Seuil par défaut (0.5)')

plt.xlabel("Seuil de décision (Threshold de Probabilité)")
plt.ylabel("Score Metrics")
plt.title("Arbitrage Précision / Recall selon le seuil de décision")
plt.legend(loc="lower left")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# 2. Affichage d'un tableau d'aide à la décision (Simulation Métier)
thresholds_to_check = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
print("========================================")
print("   SIMULATION : IMPACT DU SEUIL DÉCISION")
print("========================================")
for t in thresholds_to_check:
    # Trouver l'indice du seuil le plus proche de la cible étudiée
    idx = np.argmin(np.abs(thresholds - t))
    print(f"Seuil ~{thresholds[idx]:.2f} ➔ Précision: {precisions[idx]:.2%} | Recall: {recalls[idx]:.2%}")

In [ ]:
# Feature Importance du modèle final
importances_final = pd.Series(best_model.feature_importances_,
                               index=test_X_selected.columns)
importances_final = importances_final.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances_final.plot(kind='barh', color='steelblue')
plt.title("Feature Importance — Modèle Final")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Values
import shap

# 1. Calculer les SHAP values
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(test_X_selected)

# 2. Afficher le plot (Correction de l'erreur de matrice)
plt.figure(figsize=(10, 8))

# Si shap_values est une liste (souvent avec LightGBM), on prend l'index 1
# Sinon, on prend directement shap_values
values_to_plot = shap_values[1] if isinstance(shap_values, list) else shap_values

shap.summary_plot(values_to_plot, test_X_selected, show=False)
plt.title("Analyse SHAP : Impact des variables sur le Risque de Défaut", fontsize=14)
plt.show()

## Interprétation de l'expertise modèle via les valeurs SHAP

L'utilisation de la méthode **SHAP (SHapley Additive exPlanations)** permet de décomposer la logique décisionnelle du modèle LightGBM. Plutôt que de traiter l'algorithme comme une boîte noire, nous isolons l'influence de chaque variable sur la probabilité finale de défaut de paiement.

### Guide de lecture du Summary Plot
Chaque point sur le graphique représente une observation du jeu de données :
* **Impact (Axe X)** : Une valeur SHAP positive indique que la variable augmente le risque de défaut. Une valeur négative indique une réduction du risque.
* **Intensité (Couleur)** : Le dégradé du bleu au rouge représente le passage d'une valeur faible à une valeur élevée pour la variable concernée.

---

### 1. Principaux indicateurs de risque identifiés

L'analyse fait ressortir trois facteurs majeurs qui dégradent la solvabilité des dossiers :

* **La maturité du prêt (term_60_months)** : C'est le prédicteur le plus discriminant. L'engagement sur 60 mois augmente structurellement le risque de défaut. La durée prolongée accroît l'exposition aux aléas financiers et réduit la visibilité sur la capacité de remboursement à long terme.
* **La pression des mensualités (installment)** : Des échéances lourdes pèsent directement sur le budget de l'emprunteur. Le modèle identifie une corrélation nette entre l'augmentation des mensualités et la probabilité de défaillance.
* **Le taux d'utilisation du crédit (bc_util & dti)** : Un ratio d'endettement élevé ou une utilisation intensive des lignes de crédit disponibles signalent un stress financier immédiat. Ces variables capturent la dépendance de l'emprunteur au crédit pour sa gestion courante.

### 2. Facteurs de solvabilité et mécanismes de protection

À l'inverse, certains indicateurs agissent comme des garanties de fiabilité :

* **Le score FICO (fico_avg)** : L'historique de paiement reste le pilier de la confiance bancaire. Les scores élevés sont systématiquement associés à une réduction du risque, confirmant la pertinence de ce scoring externe.
* **Le paradoxe du montant emprunté (loan_amnt)** : On observe que les montants importants tendent à réduire le risque dans ce modèle spécifique. Cela reflète un biais de sélection positif : la plateforme n'accorde des sommes élevées qu'aux profils présentant des garanties supérieures.
* **L'ancienneté bancaire (credit_age_years)** : La stabilité historique est un facteur protecteur. Les profils avec un historique de crédit récent sont pénalisés par le modèle car ils représentent une incertitude statistique.

---

### 3. Efficacité des variables créées (Feature Engineering)

Les indicateurs conçus lors de la phase de préparation des données apportent une valeur ajoutée significative à la précision du modèle :

* **Ratio mensualité / revenu (installment_to_income)** : Cette variable se positionne parmi les plus influentes. Elle est plus pertinente que le revenu brut car elle mesure la charge réelle supportée par le budget mensuel de l'emprunteur.
* **Limite de crédit globale (total_bc_limit)** : Une limite élevée, souvent accordée par d'autres institutions, sert de signal de solvabilité pré-établie, ce que le modèle interprète comme un facteur de sécurité.

---

### Recommandations pour la politique d'octroi

Sur la base de ces résultats, trois axes stratégiques sont proposés :

1. **Sélectivité sur le long terme** : Revoir les critères d'éligibilité pour les prêts à 60 mois en exigeant un score FICO minimum plus élevé pour compenser le risque lié à la durée.
2. **Surveillance des seuils d'endettement** : Le modèle montre une rupture de risque dès que le taux d'endettement dépasse 30%. Ce seuil devrait constituer une alerte automatique lors de l'étude du dossier.
3. **Valorisation de l'expérience de crédit** : Intégrer un bonus de score pour les emprunteurs justifiant de plus de dix ans d'historique de crédit stable, le modèle confirmant la fiabilité de ces profils.

# Conclusions et Recommandations

Ce projet de Data Science visait à transformer les données historiques de **LendingClub** en un système d'aide à la décision robuste. En nous appuyant sur plus de deux millions d'observations, nous avons développé un modèle capable d'évaluer le risque de crédit avec une précision statistique significative.

---

### 1. Performances et Validation du Modèle Final

Le modèle **LightGBM**, optimisé par une stratégie d'échantillonnage (**Undersampling**), a été validé sur un jeu de données test afin de garantir la fiabilité des prédictions en conditions réelles.

* **AUC-ROC (0.72) et AUPRC (0.41)** : Ce score démontre une solide capacité de discrimination. Le modèle parvient à classer correctement un profil à risque au-dessus d'un profil sain dans 72 % des cas. L'AUPRC (Average Precision) de 0.41 vient confirmer la robustesse du modèle face au déséquilibre naturel des classes dans le domaine du crédit.
* **Protection du capital financier (Recall Classe 1 : 67 %)** : La priorité métier étant d'éviter les pertes, le modèle se distingue par sa capacité à identifier les futurs défauts de paiement. Il parvient à détecter 67 % des mauvais crédits, ce qui constitue un excellent rempart contre le risque d'impayés.
* **Stabilité du Scoring** : La distribution des probabilités montre une séparation nette des classes, validant l'utilisation du modèle comme outil de segmentation du risque.
* **Métrique métier orientée risque (F2-Score : 0.56)** : Dans notre contexte, un faux négatif (accorder un prêt à un client défaillant) coûte beaucoup plus cher qu'un faux positif (refuser un bon client). Le F2-Score, qui accorde deux fois plus de poids au rappel (détection des défauts) qu'à la précision, valide l'alignement du modèle avec cette stratégie défensive.
* **Équilibre Précision/Rappel (Précision Classe 1 : 34 %)** : La stratégie d'échantillonnage permet de détecter une part critique des défauts de paiement tout en limitant le taux de faux refus, préservant ainsi le volume d'activité commerciale.

---

### 2. Analyse des Leviers de Risque (Vision Métier)

L'interprétation des résultats via les valeurs SHAP identifie cinq piliers fondamentaux expliquant la solvabilité des emprunteurs :

1. **La durée d'engagement (term_60_months)** : Le passage à un prêt sur 5 ans est le facteur de risque prédominant. L'allongement de la maturité réduit la visibilité financière et augmente mécaniquement la probabilité d'incident de paiement.
2. **La tarification du risque (int_rate)** : Le taux d'intérêt agit comme un puissant indicateur avancé. Un taux élevé, s'il compense le risque, alourdit également la charge de la dette, créant un facteur de stress financier supplémentaire.
3. **Le socle de confiance (fico_avg)** : Le score de crédit demeure le prédicteur le plus stable. Il reflète la discipline de paiement historique, qui s'avère être le meilleur rempart contre le défaut futur.
4. **La pression budgétaire (installment_to_income)** : Ce ratio, créé lors de notre phase d'ingénierie des variables, démontre que le risque est moins lié au revenu brut qu'à la part de ce revenu absorbée par la mensualité.
5. **L'expérience de crédit (credit_age_years)** : L'ancienneté bancaire est synonyme de stabilité. Les profils avec un historique réduit (moins de 5 ans) présentent une volatilité comportementale plus élevée.

---

### 3. Recommandations Stratégiques

Pour optimiser la rentabilité du portefeuille et maîtriser le coût du risque, nous préconisons les mesures suivantes :

* **Automatisation des seuils d'octroi** : Instaurer des filtres stricts sur le taux d'endettement (DTI) au-delà de 35 % et sur le ratio mensualité/revenu dès qu'il dépasse 15 %.
* **Ajustement de la politique tarifaire** : Renforcer les exigences de fonds propres et ajuster les taux pour les prêts de 60 mois afin de couvrir le risque structurel lié à la durée.
* **Encadrement des nouveaux profils** : Pour les emprunteurs disposant d'un historique de crédit court (moins de 3 ans), limiter les montants accordés (ex: plafond à 10 000 $) pour tester leur capacité de remboursement avant d'augmenter l'exposition.

---

### 4. Limites et Perspectives d'Évolution

Bien que performant, le modèle actuel constitue une base qui peut être enrichie :

* **Contexte macroéconomique** : Les données utilisées (2007-2018) ne reflètent pas les chocs récents (inflation, remontée des taux directeurs). L'intégration d'indicateurs économiques actualisés permettrait d'ajuster les prédictions en temps réel.
* **Enrichissement des données** : L'ajout de variables sur la stabilité de l'emploi (secteur d'activité, ancienneté dans le poste) renforcerait la précision du profilage.
* **Évolutivité technique** : Le déploiement du modèle via une API permettrait de réaliser des prédictions instantanées dès la saisie du dossier, offrant une réactivité optimale pour le service d'octroi.